<div dir="rtl" style="text-align: right;">
<h1><b>למידת מכונה - פרויקט</b></h1>
</div>

<div dir="rtl">
</div>
<div dir="rtl">
<h2>
<b>חלק 1 - הקדמה:</b> </h2>

פרטי הסטודנטים:
נויה א 6233, עמית א 9054 ואמיר מ 8889.

שימוש בכלי AI (פרומפטים):
נעזרנו בעוזר AI (Claude, דרך Claude Code) לאורך הפרויקט. הפרומפטים העיקריים:
- בקשה לעזור להקים פרויקט למידת מכונה על סמך קובץ ההנחיות וה- dataset שנבחר, כולל המלצה איזה אלגוריתם מבין הנלמדים בקורס הכי מתאים לבעיית סיווג טקסט (קיבלנו המלצה מנומקת ל- Naive Bayes)
- בקשה לבנות את שלד המחברת: טעינת הנתונים, feature engineering (Bag of Words), מימוש Naive Bayes מאפס (fit/ predict), אימון והערכה על ה-test
- בקשה להוסיף את חלק ההרחבה (6): grid-search עם k fold cross validation לבחירת feature engineering ו- alpha, טיפול בחוסר איזון בין הרגשות (oversampling), והסבר (explainability) על סמך מה שהמודל למד
- בקשה להוסיף תא לבדיקה ידנית - הזנת משפט וקבלת הרגש החזוי
- מספר בקשות לוודא שהמחברת עומדת בדרישות הספציפיות של מסמך ההנחיות, ותיקונים בהתאם
בכל שלב בדקנו את התוצאות בעצמנו והבנו את הקוד וההיגיון מאחוריו לפני שהמשכנו הלאה.

הסבר על בעיית הלמידה וה- dataset:
בחרנו ב- dataset מתוך Kaggle בשם "Emotions Detection". מסד נתונים זה מכיל משפטים קצרים ומתייג כל אחד מהם לרגש מסוים. המטרה היא לאמן מודל למידת מכונה שיוכל לסווג טקסטים גולמיים לרגש המתאים להם. מדובר בבעיית סיווג רב-מחלקתית (Multi class Classification) בתחום ניתוח טקסט (NLP).

</div>

<div dir="rtl" style="text-align: right;">
<h3>הבעיה וה- Dataset:</h3>
<p>המטלה עוסקת בבעיית סיווג רב מחלקתי  בתחום ניתוח הטקסט (NLP): בהינתן משפט יוחזר הרגש שנובע ממנו.

 ה- Dataset המקורי (<code>emotions-detection-text-dataset</code>) מכיל משפטים באנגלית שכל אחד מתויג ברגש אחד מתוך שש קטגוריות:
 anger, fear, joy, love, sadness, surprise.

  בפורמט <code>text;emotion</code>. מכיוון שהקובץ המקורי מגיע כקובץ יחיד ללא חלוקת train/ test מובנית, ביצענו חלוקה חד פעמית ל-train ו-test מיד בשלב הטעינה.  </p>
</div>

<div dir="rtl" style="text-align: right;">
<h3>טעינת ה- Dataset:</h3>
</div>

In [ ]:
%pip install -q kagglehub

import kagglehub
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 100)

Note: you may need to restart the kernel to use updated packages.


In [ ]:

dataset_path = kagglehub.dataset_download("abhrajaiswal/emotions-detection-text-dataset")

df = pd.read_csv(f"{dataset_path}/emotions.txt", sep=";", names=["text", "emotion"])
print(f"Total rows in original dataset: {len(df)}")
df.head()

Total rows in original dataset: 16000


,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned hopeful just from being around someone who cares ...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplace i will know that it is still on the property,love
4,i am feeling grouchy,anger


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["emotion"], random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train set: {len(train_df)} rows | Test set: {len(test_df)} rows")

Train set: 12800 rows | Test set: 3200 rows


In [ ]:
print("First 5 rows of the Train set:")
display(train_df.head())

First 5 rows of the Train set:


,text,emotion
0,i was feeling drained before i even sat in the chair,sadness
1,i alsways feel so carefree,joy
2,i dont know about you guys but i certainly feel fabulous about myself,joy
3,i also learned that when i feel passionate about what i m writing i can actually be quite good a...,joy
4,i feel like im loving them even more now that im working again i appreciate every snuggle and fe...,love


In [ ]:
print("First 5 rows of the Test set:")
display(test_df.head())

First 5 rows of the Test set:


,text,emotion
0,i feel wronged by certain people and my instinct was to get angry at them and stop speaking to t...,anger
1,i feel so calm with the routine rinse wash with detergent rinse take outside to line dry,joy
2,i feel like im not welcomed here i just dont like blend in or something,joy
3,i feel totally listless exams have come and gone and now i have a whole five or so months in fro...,sadness
4,i am feeling confident that i will be able to get to the back door before dinner time,joy


<div dir="rtl" style="text-align: right;">
<h2><b>מדד האיכות:</b></h2>
<p>מדובר בבעיית סיווג רב מחלקתי  עם 6 מחלקות (anger, fear, joy, love, sadness, surprise) ללא מחלקה מרכזית אחת. לכן, לפי הנחיות המטלה, מדד האיכות בו נשתמש הוא macro average F1  ממוצע (לא משוקלל) של ציון ה- F1 שמחושב בנפרד לכל מחלקה. בחירה זו מתאימה כיוון שהיא נותנת משקל שווה לכל רגש, גם לרגשות נדירים יחסית ב-dataset (כמו surprise ו- love), ולא רק לרגשות הנפוצים (כמו joy ו- sadness).</p>
</div>

In [ ]:
from sklearn.metrics import f1_score, classification_report

def evaluate(y_true, y_pred, title=""):
    score = f1_score(y_true, y_pred, average="macro")
    print(f"{title} macro-F1: {score:.4f}")
    return score

<div dir="rtl" style="text-align: right;">
<h2><b>חלק 2 - Feature Engineering:</b></h2>
<p>הטקסט לא קביל ישירות לאלגוריתם למידה, לכן נהפוך כל משפט לוקטור מספרי בשיטת Bag of Words (וקטוריזציה על בסיס ספירת מילים) שנלמדה בכיתה: כל מאפיין  מייצג מילה מהאוצר מילים , והערך שלו הוא מספר הפעמים שהמילה מופיעה במשפט. אנו מסננים מילות עצירה נפוצות  שאינן נושאות מידע על הרגש (כגון "the", "is", "and"). זהו הבסיס הטבעי לאלגוריתם Naive Bayes המשתמש בשכיחויות מילים לכל מחלקה.</p>
<p>ה-vectorizer מותאם (fit) רק על ה- trainset, ולאחר מכן משמש להמרת (transform) גם את ה- trainset וגם את ה- testset  כך נמנעים מדליפת מידע מה- testset.</p>
</div>

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])
y_train = train_df["emotion"].values
y_test = test_df["emotion"].values

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} words")
print(f"Train matrix shape: {X_train.shape} | Test matrix shape: {X_test.shape}")

Vocabulary size: 13225 words
Train matrix shape: (12800, 13225) | Test matrix shape: (3200, 13225)


<div dir="rtl" style="text-align: right;">
<h3>הדגמת ה- Feature Engineering על דוגמאות:</h3>
</div>

In [ ]:
def show_bow_example(text_series, X, idx):
    row = X[idx]
    words = vectorizer.get_feature_names_out()[row.indices]
    counts = row.data
    print(f"Original sentence: {text_series.iloc[idx]!r}")
    print("Bag-of-Words representation (word: count):", dict(zip(words, counts)))
    print()

print("--- Train examples ---")
for i in [0, 1, 2]:
    show_bow_example(train_df["text"], X_train, i)

print("--- Test examples ---")
for i in [0, 1]:
    show_bow_example(test_df["text"], X_test, i)

--- Train examples ---


Original sentence: 'i was feeling drained before i even sat in the chair'
Bag-of-Words representation (word: count): {'feeling': np.int64(1), 'drained': np.int64(1), 'sat': np.int64(1), 'chair': np.int64(1)}

Original sentence: 'i alsways feel so carefree'
Bag-of-Words representation (word: count): {'alsways': np.int64(1), 'feel': np.int64(1), 'carefree': np.int64(1)}

Original sentence: 'i dont know about you guys but i certainly feel fabulous about myself'
Bag-of-Words representation (word: count): {'feel': np.int64(1), 'dont': np.int64(1), 'know': np.int64(1), 'guys': np.int64(1), 'certainly': np.int64(1), 'fabulous': np.int64(1)}

--- Test examples ---
Original sentence: 'i feel wronged by certain people and my instinct was to get angry at them and stop speaking to them but two wrongs dont make a right i think'
Bag-of-Words representation (word: count): {'angry': np.int64(1), 'certain': np.int64(1), 'dont': np.int64(1), 'feel': np.int64(1), 'instinct': np.int64(1), 'make': np.int6

<div dir="rtl" style="text-align: right;">
<h2><b>חלק 3 - מימוש אלגוריתם למידה Multinomial Naive Bayes:</b></h2>
<p>בחרנו ב Naive Bayes כי הוא אלגוריתם גנרטיבי ומתאים באופן טבעי לנתוני טקסט המיוצגים כספירות מילים (Bag of Words): הוא מניח  שכל מילה במשפט תורמת עדות בלתי תלויה לרגש שלו, ומחשב לפי חוק בייס את ההסתברות של כל מחלקה (רגש) בהינתן המילים שבמשפט.</p>
<p><b>איך האלגוריתם עובד:</b></p>
<ul>
<li><b>שלב האימון (<code>fit</code>):</b> עבור כל רגש (מחלקה), סופרים כמה פעמים כל מילה מופיעה בכל המשפטים שמתויגים באותו רגש, ומוסיפים לכל ספירה את הפרמטר <code>alpha</code> (ה- smoothing). כך מתקבלת לכל רגש "טביעת אצבע" של שכיחות מילים. בנוסף, שומרים כמה שכיח כל רגש בכלל ב-trainset (ה- prior).</li>
<li><b>שלב החיזוי (<code>predict</code>):</b> עבור משפט חדש, מחשבים לכל רגש ציון שמשלב את ה- prior שלו יחד עם עד כמה המילים שבמשפט "מתאימות" לטביעת האצבע של אותו רגש (כלומר, כמה שכיחות המילים במשפט תואמות למה שנצפה בעבר עבור אותו רגש). הרגש עם הציון הגבוה ביותר הוא זה שנבחר כתשובת המודל.</li>
</ul>
<p><b>Hyperparameter בו השתמשנו:</b> <code>alpha</code> - מקדם ה-Laplace smoothing , שמטפל ב"בעיית השכיחות אפס" - מילה שלא הופיעה כלל באימון עבור מחלקה מסוימת לא צריכה לאפס את כל ההסתברות שלה. ככל ש- alpha גדול יותר, כך ההסתברויות "מוחלקות" יותר לכיוון אחיד בין המילים.</p>
</div>

In [ ]:
import numpy as np

class MultinomialNaiveBayes:

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        n_docs, n_features = X.shape
        self.class_log_prior_ = np.zeros(len(self.classes_))
        self.feature_log_prob_ = np.zeros((len(self.classes_), n_features))
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_log_prior_[i] = np.log(X_c.shape[0] / n_docs)
            word_counts = np.asarray(X_c.sum(axis=0)).flatten() + self.alpha
            self.feature_log_prob_[i] = np.log(word_counts / word_counts.sum())
        return self

    def predict_log_proba(self, X):
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict(self, X):
        log_proba = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_proba, axis=1)]

<div dir="rtl" style="text-align: right;">
<h3>בדיקת תקינות:</h3>
<p>נוודא שהמימוש העצמי שלנו מניב תוצאות זהות (או קרובות מאוד) למימוש הבנוי מראש של <code>sklearn.naive_bayes.MultinomialNB</code>, כאשר משתמשים באותו alpha.</p>
</div>

In [ ]:
from sklearn.naive_bayes import MultinomialNB

_our_model = MultinomialNaiveBayes(alpha=1.0).fit(X_train, y_train)
_sklearn_model = MultinomialNB(alpha=1.0).fit(X_train, y_train)

_our_pred = _our_model.predict(X_test)
_sklearn_pred = _sklearn_model.predict(X_test)

agreement = (_our_pred == _sklearn_pred).mean()
print(f"Agreement between our implementation and sklearn: {agreement:.2%}")
assert agreement > 0.99, "Our implementation deviates significantly from sklearn - check the code"

Agreement between our implementation and sklearn: 100.00%


<div dir="rtl" style="text-align: right;">
<h2><b>חלק 4 - אימון: הפעלת ה- flow לפי פרמטרים שונים:</b></h2>
<p>פונקציית ה-<code>fit</code> שממומשת למעלה תומכת בכל ערך של alpha. להלן דוגמה להרצת האימון עם כמה ערכי alpha שונים, ולבחירת המודל הסופי שיאומן על כל ה-trainset.</p>
<p style="color:#b00;">הערה: ההשוואה השיטתית והמלאה של hyperparameters (grid search + k fold cross validation), האימון מחדש עם הפרמוטציה המנצחת, ודוגמאות ה- feature engineering (Train ו-Test)</b> - מופיעים בהמשך, בסעיף ההרחבה 6.א.</p>
</div>

In [ ]:
for alpha_try in [0.1, 1.0, 5.0]:
    model_try = MultinomialNaiveBayes(alpha=alpha_try).fit(X_train, y_train)
    train_pred = model_try.predict(X_train)
    evaluate(y_train, train_pred, title=f"alpha={alpha_try} | train")

final_alpha = 1.0
model = MultinomialNaiveBayes(alpha=final_alpha).fit(X_train, y_train)
print(f"\nFinal model trained with alpha={final_alpha} on {X_train.shape[0]} training examples")

alpha=0.1 | train macro-F1: 0.9616
alpha=1.0 | train macro-F1: 0.8373
alpha=5.0 | train macro-F1: 0.4666

Final model trained with alpha=1.0 on 12800 training examples


<div dir="rtl" style="text-align: right;">
<h2><b>חלק 5 - חיזוי ושערוך איכות המודל על ה- test set:</b></h2>
<p>המודל שאומן בחלק 4 מפעיל <code>predict</code> על ה- test set (לאחר אותו preprocessing שהוגדר ב- vectorizer בחלק 2 ללא אימון נוסף עליו). מציגים את 5 החיזויים הראשונים לצד הרגש האמיתי, ומודדים את איכות המודל לפי מדד macro F1 שנקבע למעלה, יחד עם דוח מפורט (precision/recall/F1) לכל רגש בנפרד.</p>
</div>

In [ ]:
test_predictions = model.predict(X_test)

print("First 5 predictions on the test set:")
for i in range(5):
    print(f"  Sentence: {test_df['text'].iloc[i]!r}")
    print(f"  True emotion: {y_test[i]}  |  Predicted emotion: {test_predictions[i]}")
    print()

First 5 predictions on the test set:
  Sentence: 'i feel wronged by certain people and my instinct was to get angry at them and stop speaking to them but two wrongs dont make a right i think'
  True emotion: anger  |  Predicted emotion: anger

  Sentence: 'i feel so calm with the routine rinse wash with detergent rinse take outside to line dry'
  True emotion: joy  |  Predicted emotion: joy

  Sentence: 'i feel like im not welcomed here i just dont like blend in or something'
  True emotion: joy  |  Predicted emotion: joy

  Sentence: 'i feel totally listless exams have come and gone and now i have a whole five or so months in front of me with no uni and free time'
  True emotion: sadness  |  Predicted emotion: sadness

  Sentence: 'i am feeling confident that i will be able to get to the back door before dinner time'
  True emotion: joy  |  Predicted emotion: joy



In [ ]:
test_score = evaluate(y_test, test_predictions, title="Test set")
print()
print(classification_report(y_test, test_predictions))

Test set macro-F1: 0.6331



              precision    recall  f1-score   support

       anger       0.87      0.66      0.75       432
        fear       0.86      0.64      0.74       387
         joy       0.75      0.93      0.83      1072
        love       0.89      0.33      0.48       261
     sadness       0.75      0.93      0.83       933
    surprise       0.73      0.10      0.17       115

    accuracy                           0.78      3200
   macro avg       0.81      0.60      0.63      3200
weighted avg       0.79      0.78      0.76      3200



<div dir="rtl" style="text-align: right;">
<h2><b>חלק 6 - הרחבה:</b></h2>
<p>בסעיף זה מתנסים בשלושה כיווני הרחבה: (א) grid search + k fold cross validation על feature engineering ו- hyperparameters, (ב) התמודדות עם data imbalanced, (ג) הסבר של מה שהמודל למד.</p>
</div>

<div dir="rtl" style="text-align: right;">
<h2><b>6.א - Grid Search + K Fold Cross Validation:</b></h2>
<p>מתנסים בשתי שיטות וקטוריזציה (Bag of Words מול TF IDF) ובכמה ערכי <code>alpha</code> - grid search על כל הפרמוטציות. עבור כל פרמוטציה מריצים 5 fold cross validation על ה- trainset: מאמנים (כולל fit של ה- vectorizer) על 4 חלקים ובודקים macro F1 על החלק החמישי, וחוזרים על כך 5 פעמים ולוקחים ממוצע.</p>
</div>

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

feature_variants = {
    "Bag-of-Words (CountVectorizer)": CountVectorizer,
    "TF-IDF (TfidfVectorizer)": TfidfVectorizer,
}
alpha_values = [0.1, 0.5, 1.0, 2.0]

train_text = train_df["text"].values
train_labels = train_df["emotion"].values
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_results = []
for feat_name, VectorizerClass in feature_variants.items():
    for alpha in alpha_values:
        fold_scores = []
        for fold_train_idx, fold_val_idx in kfold.split(train_text, train_labels):
            fold_vectorizer = VectorizerClass(stop_words="english")
            X_fold_train = fold_vectorizer.fit_transform(train_text[fold_train_idx])
            X_fold_val = fold_vectorizer.transform(train_text[fold_val_idx])
            fold_model = MultinomialNaiveBayes(alpha=alpha).fit(X_fold_train, train_labels[fold_train_idx])
            fold_pred = fold_model.predict(X_fold_val)
            fold_scores.append(f1_score(train_labels[fold_val_idx], fold_pred, average="macro"))
        grid_results.append({
            "feature_engineering": feat_name,
            "alpha": alpha,
            "mean_cv_macro_f1": np.mean(fold_scores),
        })

grid_results_df = pd.DataFrame(grid_results).sort_values("mean_cv_macro_f1", ascending=False).reset_index(drop=True)
grid_results_df

,feature_engineering,alpha,mean_cv_macro_f1
0,Bag-of-Words (CountVectorizer),0.1,0.693970
1,Bag-of-Words (CountVectorizer),0.5,0.674142
2,TF-IDF (TfidfVectorizer),0.1,0.609101
3,Bag-of-Words (CountVectorizer),1.0,0.606825
4,Bag-of-Words (CountVectorizer),2.0,0.495511
5,TF-IDF (TfidfVectorizer),0.5,0.485055
6,TF-IDF (TfidfVectorizer),1.0,0.411589
7,TF-IDF (TfidfVectorizer),2.0,0.335930


In [ ]:
best_combo = grid_results_df.iloc[0]
best_feature_name = best_combo["feature_engineering"]
best_alpha = float(best_combo["alpha"])

print(f"Best permutation: {best_feature_name}, alpha={best_alpha} "
      f"-> mean CV macro-F1 = {best_combo['mean_cv_macro_f1']:.4f}")

Best permutation: Bag-of-Words (CountVectorizer), alpha=0.1 -> mean CV macro-F1 = 0.6940


<div dir="rtl" style="text-align: right;">
<h3>אימון סופי עם הפרמוטציה המנצחת, על כל ה- trainset:</h3>
<p>לפי הפרמוטציה שנבחרה, מאמנים מחדש (הן את ה- vectorizer והן את המודל) על כל ה- trainset, ומבצעים את אותו preprocessing גם על ה- testset.</p>
</div>

In [ ]:
BestVectorizerClass = feature_variants[best_feature_name]

vectorizer = BestVectorizerClass(stop_words="english")
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

final_alpha = best_alpha
model = MultinomialNaiveBayes(alpha=final_alpha).fit(X_train, y_train)
print(f"Final model retrained with {best_feature_name}, alpha={final_alpha}, "
      f"on {X_train.shape[0]} training examples")

Final model retrained with Bag-of-Words (CountVectorizer), alpha=0.1, on 12800 training examples


In [ ]:
def show_feature_vector_example(text_series, X, idx):
    row = X[idx]
    words = vectorizer.get_feature_names_out()[row.indices]
    values = row.data
    print(f"Sentence: {text_series.iloc[idx]!r}")
    print(f"Feature vector ({best_feature_name}, word: value):", dict(zip(words, values)))
    print()

print("--- Train examples through the winning feature engineering ---")
for i in [0, 1, 2]:
    show_feature_vector_example(train_df["text"], X_train, i)

--- Train examples through the winning feature engineering ---
Sentence: 'i was feeling drained before i even sat in the chair'
Feature vector (Bag-of-Words (CountVectorizer), word: value): {'feeling': np.int64(1), 'drained': np.int64(1), 'sat': np.int64(1), 'chair': np.int64(1)}

Sentence: 'i alsways feel so carefree'
Feature vector (Bag-of-Words (CountVectorizer), word: value): {'alsways': np.int64(1), 'feel': np.int64(1), 'carefree': np.int64(1)}

Sentence: 'i dont know about you guys but i certainly feel fabulous about myself'
Feature vector (Bag-of-Words (CountVectorizer), word: value): {'feel': np.int64(1), 'dont': np.int64(1), 'know': np.int64(1), 'guys': np.int64(1), 'certainly': np.int64(1), 'fabulous': np.int64(1)}



In [ ]:
print("--- Test examples through the winning feature engineering ---")
for i in [0, 1]:
    show_feature_vector_example(test_df["text"], X_test, i)

--- Test examples through the winning feature engineering ---
Sentence: 'i feel wronged by certain people and my instinct was to get angry at them and stop speaking to them but two wrongs dont make a right i think'
Feature vector (Bag-of-Words (CountVectorizer), word: value): {'angry': np.int64(1), 'certain': np.int64(1), 'dont': np.int64(1), 'feel': np.int64(1), 'instinct': np.int64(1), 'make': np.int64(1), 'people': np.int64(1), 'right': np.int64(1), 'speaking': np.int64(1), 'stop': np.int64(1), 'think': np.int64(1), 'wronged': np.int64(1)}

Sentence: 'i feel so calm with the routine rinse wash with detergent rinse take outside to line dry'
Feature vector (Bag-of-Words (CountVectorizer), word: value): {'calm': np.int64(1), 'dry': np.int64(1), 'feel': np.int64(1), 'line': np.int64(1), 'outside': np.int64(1), 'routine': np.int64(1), 'wash': np.int64(1)}



In [ ]:
test_predictions = model.predict(X_test)

print("First 5 predictions on the test set (after 6.a tuning):")
for i in range(5):
    print(f"  Sentence: {test_df['text'].iloc[i]!r}")
    print(f"  True emotion: {y_test[i]}  |  Predicted emotion: {test_predictions[i]}")
    print()

evaluate(y_test, test_predictions, title="Test set (after 6.a tuning)")
print()
print(classification_report(y_test, test_predictions))

First 5 predictions on the test set (after 6.a tuning):
  Sentence: 'i feel wronged by certain people and my instinct was to get angry at them and stop speaking to them but two wrongs dont make a right i think'
  True emotion: anger  |  Predicted emotion: anger

  Sentence: 'i feel so calm with the routine rinse wash with detergent rinse take outside to line dry'
  True emotion: joy  |  Predicted emotion: joy

  Sentence: 'i feel like im not welcomed here i just dont like blend in or something'
  True emotion: joy  |  Predicted emotion: joy

  Sentence: 'i feel totally listless exams have come and gone and now i have a whole five or so months in front of me with no uni and free time'
  True emotion: sadness  |  Predicted emotion: sadness

  Sentence: 'i am feeling confident that i will be able to get to the back door before dinner time'
  True emotion: joy  |  Predicted emotion: joy

Test set (after 6.a tuning) macro-F1: 0.7102

              precision    recall  f1-score   support

  

<div dir="rtl" style="text-align: right;">
<h2><b>6.ב - התמודדות עם Data Imbalanced:</b></h2>
<p>כפי שראינו בדוח הסיווג, המחלקות <code>surprise</code> ו-<code>love</code> נדירות משמעותית לעומת <code>joy</code> ו-<code>sadness</code>, מה שפגע ב- recall שלהן. מבצעים random over sampling: לוקחים מדגם (עם החזרה) מכל מחלקה עד לגודל המחלקה הגדולה ביותר ב- trainset, כך שהמודל רואה כל רגש בתדירות שווה באימון. ה- testset נשאר ללא שינוי, הוא ממשיך לשקף את ההתפלגות האמיתית של הנתונים, ורק עליו משערכים את האיכות.</p>
</div>

In [ ]:
max_class_size = train_df["emotion"].value_counts().max()

oversampled_parts = []
for emotion_label, group in train_df.groupby("emotion"):
    oversampled_parts.append(group.sample(n=max_class_size, replace=True, random_state=42))
train_df_balanced = pd.concat(oversampled_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("Class distribution before oversampling:")
print(train_df["emotion"].value_counts())
print("\nClass distribution after oversampling:")
print(train_df_balanced["emotion"].value_counts())

Class distribution before oversampling:
emotion
joy         4290
sadness     3733
anger       1727
fear        1550
love        1043
surprise     457
Name: count, dtype: int64

Class distribution after oversampling:
emotion
sadness     4290
love        4290
joy         4290
anger       4290
surprise    4290
fear        4290
Name: count, dtype: int64


In [ ]:
vectorizer = BestVectorizerClass(stop_words="english")
X_train = vectorizer.fit_transform(train_df_balanced["text"])
X_test = vectorizer.transform(test_df["text"])
y_train = train_df_balanced["emotion"].values

model = MultinomialNaiveBayes(alpha=final_alpha).fit(X_train, y_train)
test_predictions = model.predict(X_test)

evaluate(y_test, test_predictions, title="Test set (after oversampling)")
print()
print(classification_report(y_test, test_predictions))

Test set (after oversampling) macro-F1: 0.6653

              precision    recall  f1-score   support

       anger       0.64      0.72      0.67       432
        fear       0.61      0.68      0.64       387
         joy       0.83      0.77      0.80      1072
        love       0.55      0.61      0.58       261
     sadness       0.80      0.76      0.78       933
    surprise       0.49      0.55      0.52       115

    accuracy                           0.73      3200
   macro avg       0.65      0.68      0.67      3200
weighted avg       0.74      0.73      0.73      3200



<div dir="rtl" style="text-align: right;">
<h2><b>6.ג - Explainability:</b></h2>
<p>המודל שלנו (Multinomial Naive Bayes) שומר, כתהליך האימון את <code>feature_log_prob_</code>  לוג ההסתברות של כל מילה בהינתן כל מחלקה. אפשר להשתמש במידע הזה (שכבר קיים במודל, בלי צורך בספריה נוספת) כדי להבין אילו מילים "הכי מזהות" כל רגש: לכל מילה ומחלקה, משווים את ה- log probability שלה באותה מחלקה לממוצע שלה בשאר המחלקות (log odds), המילים עם ההפרש הגבוה ביותר הן המילים המאפיינות ביותר את הרגש הזה.</p>
</div>

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
n_top = 8

print("Most informative words per emotion:\n")
for i, emotion_label in enumerate(model.classes_):
    others_mean = (model.feature_log_prob_.sum(axis=0) - model.feature_log_prob_[i]) / (len(model.classes_) - 1)
    log_odds = model.feature_log_prob_[i] - others_mean
    top_idx = np.argsort(log_odds)[::-1][:n_top]
    print(f"{emotion_label}: {', '.join(feature_names[top_idx])}")

Most informative words per emotion:

anger: envious, rebellious, resentful, insulted, offended, irritable, dissatisfied, heartless
fear: apprehensive, pressured, shaky, frantic, distressed, suspicious, uncertain, reluctant
joy: superior, energetic, festive, clever, invigorated, smug, honored, intelligent
love: sympathetic, horny, loyal, fond, compassionate, tender, devoted, treasured
sadness: punished, deprived, disturbed, abused, disheartened, humiliated, regretful, unimportant
surprise: impressed, dazed, curious, amazed, shocked, stunned, enthralled, ludicrous


<div dir="rtl" style="text-align: right;">
<h2><b>בדיקה ידנית - חיזוי על משפט חופשי:</b></h2>
<p>כדי לבדוק את המודל על משפט משלכם: הרצתם את כל התאים למעלה (כדי שה-<code>vectorizer</code> וה-<code>model</code> יהיו מאומנים), ואז הריצו את התא הבא, הוא יבקש מכם להקליד משפט ויחזיר את הרגש שהמודל חוזה עבורו.</p>
</div>

In [ ]:
def predict_emotion(sentence: str) -> str:
    X = vectorizer.transform([sentence])
    return model.predict(X)[0]

for example in [
    "I am so excited and happy about this",
    "I am terrified of what might happen",
    "I feel so furious right now I want to punch a wall",
]:
    print(f"{example!r} -> {predict_emotion(example)}")

'I am so excited and happy about this' -> joy
'I am terrified of what might happen' -> fear
'I feel so furious right now I want to punch a wall' -> anger


In [ ]:
sentence = input("הקלידו משפט באנגלית: ")
print(f"Predicted emotion: {predict_emotion(sentence)}")